##Silver Layer — Data Quality Investigation

#####Root-cause analysis of the `_dq_warnings` flags raised by `03_silver_flights`.
#####Every query here is reproducible against `airline_analytics.silver.flights`.

## 1. The AIRTIME_EXCEEDS_ELAPSED warning

`air_time_min` exceeds `actual_elapsed_min` on 24,070 rows in 2005 and zero rows in 2019.
Elapsed time is gate-to-gate and includes both taxi phases, so air time cannot exceed it.

### 1.1 Is it rounding, or something structural?

In [0]:
%sql
SELECT month, COUNT(*) AS bad_rows,
       MIN(air_time_min - actual_elapsed_min) AS min_gap,
       MAX(air_time_min - actual_elapsed_min) AS max_gap
FROM airline_analytics.silver.flights
WHERE year = 2005 AND array_contains(_dq_warnings, 'AIRTIME_EXCEEDS_ELAPSED')
GROUP BY month ORDER BY month

**Not rounding.** Gaps run 2–23 min at minimum and ~169 max, far beyond rounding error.
Volume tracks seasonal flight counts (~1,600–2,300/month), so it's a constant error *rate*.
Eight of twelve months cap at exactly 169, which points at a specific recurring pattern
rather than random corruption.

##### 1.2 Which carriers?

In [0]:
%sql
SELECT
  carrier_code,
  COUNT(*) AS bad_rows,
  ROUND(AVG(air_time_min - actual_elapsed_min), 1) AS avg_gap,
  SUM(CASE WHEN arr_time_hhmm < dep_time_hhmm THEN 1 ELSE 0 END) AS crosses_midnight,
  COUNT(DISTINCT CONCAT(origin, '-', dest)) AS distinct_routes
FROM airline_analytics.silver.flights
WHERE year = 2005 AND array_contains(_dq_warnings, 'AIRTIME_EXCEEDS_ELAPSED')
GROUP BY carrier_code
ORDER BY bad_rows DESC

**Two carriers, 99.3% of rows.** JetBlue 19,390 and Hawaiian 4,509 out of 24,070; everything
else is single or double digits. Small route sets (47 and 12), average gaps of 141 and 136 min.
Both were young carriers in 2005 running mostly long-haul west-coast routes.

##### 1.3 Which field is actually broken?

In [0]:
%sql
SELECT carrier_code, origin, dest, COUNT(*) AS n,
       ROUND(AVG(crs_elapsed_min), 0) AS avg_scheduled,
       ROUND(AVG(actual_elapsed_min), 0) AS avg_actual,
       ROUND(AVG(air_time_min), 0) AS avg_air
FROM airline_analytics.silver.flights
WHERE year = 2005 AND array_contains(_dq_warnings, 'AIRTIME_EXCEEDS_ELAPSED')
  AND carrier_code IN ('B6','HA')
GROUP BY carrier_code, origin, dest
ORDER BY n DESC LIMIT 15

**`AirTime` is broken; `ActualElapsedTime` is fine.** avg_actual tracks avg_scheduled almost
exactly (LGB→JFK: 310 vs 310), while avg_air is ~160 min higher.

The excess scales with time zones crossed:
- 3 zones (LGB/OAK/LAS/BUR/SAN/ONT→JFK, HNL→LAX): ~160 min
- 2 zones (DEN→JFK): ~103 min
- 1 zone (MSY→JFK): ~41 min

Both carriers computed air time from raw local wheels-off/wheels-on clock times without
correcting for the zone change.

### 1.4 Confirming the mechanism: directional asymmetry

If the cause is an uncorrected time-zone offset, the error should appear only on **eastbound**
legs. Flying east, the offset inflates the raw clock difference and air time exceeds elapsed.
Flying west, the same offset *deflates* it — and an understated air time can never exceed
elapsed, so the warning cannot fire.

A result of zero westbound rows here is the finding, not a broken query.

In [0]:
%sql
SELECT origin, dest, COUNT(*) AS n
FROM airline_analytics.silver.flights
WHERE year = 2005 AND carrier_code = 'B6'
  AND ((origin = 'JFK' AND dest IN ('LGB','OAK','LAS','BUR','SAN','ONT'))
    OR (dest = 'JFK' AND origin IN ('LGB','OAK','LAS','BUR','SAN','ONT')))
  AND array_contains(_dq_warnings, 'AIRTIME_EXCEEDS_ELAPSED')
GROUP BY origin, dest ORDER BY n DESC

**Perfect asymmetry.** Six westbound routes flagged (LGB→JFK 2,601 down to ONT→JFK 602),
zero eastbound. Mechanism confirmed.

### 1.5 The half the warning could not see

**No `_dq_warnings` filter here, deliberately.** JFK→west legs are broken by the same offset
in the opposite direction, so they never trip `AIRTIME_EXCEEDS_ELAPSED` and are invisible to
the query above. Detecting them requires comparing air time against implied air time
(`actual_elapsed_min - taxi_out_min - taxi_in_min`) rather than against elapsed alone.

In [0]:
%sql
SELECT origin, dest, COUNT(*) AS n,
       ROUND(AVG(air_time_min), 0) AS avg_air,
       ROUND(AVG(actual_elapsed_min - taxi_out_min - taxi_in_min), 0) AS implied_air,
       ROUND(AVG(actual_elapsed_min - taxi_out_min - taxi_in_min - air_time_min), 0) AS shortfall
FROM airline_analytics.silver.flights
WHERE year = 2005 AND carrier_code = 'B6'
  AND origin = 'JFK' AND dest IN ('LGB','OAK','LAS','BUR','SAN','ONT')
  AND air_time_min IS NOT NULL AND actual_elapsed_min IS NOT NULL
GROUP BY origin, dest ORDER BY n DESC

**Exactly 180 min short on every route** — 328−148, 342−163, 301−121, 328−148, 322−142,
321−141. Three time zones, three hours, no variance.

Row counts mirror the eastbound query almost exactly (JFK→LGB 2,601 vs LGB→JFK 2,601), so
close to 100% of B6 flights on these routes are affected in both directions.

**This is why `AIRTIME_ELAPSED_MISMATCH` was added to `SOFT_CHECKS`** — it uses the implied-air-time
residual with a 15-min tolerance and catches both directions. After adding it, 2005 rose from
24,070 flagged rows to 48,169, roughly double, as the round-trip symmetry predicts. 2019 stayed
at zero, confirming the rule doesn't fire spuriously on clean data.

## 2. A second, unrelated defect: SkyWest, 2004

2004 carries 207,046 mismatches (2.9% of rows) against 0.67% in 2005. More telling, the ratio
between the two rules breaks down: in 2005 and 2006 `AIRTIME_ELAPSED_MISMATCH` is roughly
double `AIRTIME_EXCEEDS_ELAPSED` (the round-trip symmetry above), but in 2004 it collapses to
1.1. Almost no westbound partners — so this is not the time-zone mechanism.

Note also that 2007 and 2008 show **zero** occurrences despite being the same source and the
same carriers. The defect has a start and an end date; it is not a general property of the
pre-2009 data.

In [0]:
%sql
SELECT carrier_code, COUNT(*) AS n,
       ROUND(AVG(air_time_min - actual_elapsed_min), 1) AS avg_gap,
       COUNT(DISTINCT CONCAT(origin, '-', dest)) AS routes
FROM airline_analytics.silver.flights
WHERE year = 2004 AND array_contains(_dq_warnings, 'AIRTIME_EXCEEDS_ELAPSED')
GROUP BY carrier_code ORDER BY n DESC

**A third carrier, and it doesn't fit.** SkyWest (OO): 168,473 rows across **424 routes**,
avg gap 94.4 min. A regional feeder flying short hops within one or two zones — a 94-min error
on routes often under 90 min total cannot be a time-zone offset. OO barely appears in 2005
(65 rows) or 2006, so this is 2004-only and separate from the B6/HA problem.

### 2.1 When did it start and stop?

In [0]:
%sql
SELECT month, COUNT(*) AS n,
       ROUND(AVG(air_time_min), 0) AS avg_air,
       ROUND(AVG(actual_elapsed_min), 0) AS avg_elapsed,
       ROUND(AVG(crs_elapsed_min), 0) AS avg_scheduled,
       ROUND(AVG(air_time_min - actual_elapsed_min), 0) AS avg_gap
FROM airline_analytics.silver.flights
WHERE year = 2004 AND carrier_code = 'OO'
  AND array_contains(_dq_warnings, 'AIRTIME_EXCEEDS_ELAPSED')
GROUP BY month ORDER BY month

**A switch thrown at the end of July 2004.** Jan–Jul: ~23,000 affected rows per month.
Aug–Dec: 57, 36, 34, 29, 34. Not a decline — a stop.

`avg_elapsed` (86–89) tracks `avg_scheduled` (89–90) throughout, so elapsed time is correct.
But `avg_air` reads 180–184, roughly **double** elapsed, on short regional routes.

### 2.2 Is it a fixed multiplier?

**No `_dq_warnings` filter — fleet-wide on purpose.** Restricting to flagged rows would only
describe the rows that already failed. Running across all OO flights shows what share of the
fleet is affected. (Row counts here are ~36K/month vs ~23K flagged, which is the point.)

In [0]:
%sql
SELECT month,
       ROUND(AVG(air_time_min / NULLIF(actual_elapsed_min, 0)), 3) AS air_to_elapsed_ratio,
       ROUND(AVG(air_time_min / NULLIF(actual_elapsed_min - taxi_out_min - taxi_in_min, 0)), 3) AS air_to_implied_ratio,
       COUNT(*) AS n
FROM airline_analytics.silver.flights
WHERE year = 2004 AND carrier_code = 'OO' AND month <= 7
GROUP BY month ORDER BY month

`air_to_implied_ratio` holds at 2.17–2.19 across all seven months, barely moving. Suggestive
of a constant — but a mean cannot distinguish "every row is 2.18" from "a mixture of correct
rows and badly wrong ones." See 2.3.

### 2.3 Testing the constant-multiplier hypothesis

A stable mean across seven months is not evidence of a constant. Percentiles are.

In [0]:
%sql
SELECT month, COUNT(*) AS n,
  ROUND(PERCENTILE(air_time_min / NULLIF(actual_elapsed_min - taxi_out_min - taxi_in_min, 0), 0.05), 3) AS p05,
  ROUND(PERCENTILE(air_time_min / NULLIF(actual_elapsed_min - taxi_out_min - taxi_in_min, 0), 0.50), 3) AS median,
  ROUND(PERCENTILE(air_time_min / NULLIF(actual_elapsed_min - taxi_out_min - taxi_in_min, 0), 0.95), 3) AS p95
FROM airline_analytics.silver.flights
WHERE year = 2004 AND carrier_code = 'OO' AND month <= 7
GROUP BY month ORDER BY month

**Hypothesis rejected.** p05 is exactly 1.0 and p95 is 3.40, with the median near 2.39.
A wide spread with a hard floor at 1.0 means a substantial share of flights are perfectly
correct and the rest are corrupted by varying amounts — a **mixture**, not a uniform transform.
The 2.18 mean was the average of two populations.

This reconciles the counts: 168,473 flagged out of ~257,000 Jan–Jul flights ≈ 65%, consistent
with a median above 2 and a floor at 1.0.

**Method note:** the stable mean looked like strong evidence for a fixed multiplier and was not.
Checking the distribution before naming a mechanism is what prevented an unsupportable claim
from reaching the README.

**Hypothesis rejected.** p05 is exactly 1.0 and p95 is 3.40, with the median near 2.39.
A wide spread with a hard floor at 1.0 means a substantial share of flights are perfectly
correct and the rest are corrupted by varying amounts — a **mixture**, not a uniform transform.
The 2.18 mean was the average of two populations.

This reconciles the counts: 168,473 flagged out of ~257,000 Jan–Jul flights ≈ 65%, consistent
with a median above 2 and a floor at 1.0.

**Method note:** the stable mean looked like strong evidence for a fixed multiplier and was not.
Checking the distribution before naming a mechanism is what prevented an unsupportable claim
from reaching the README.

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN Origin = Dest THEN 1 ELSE 0 END) AS same_origin_dest,
  SUM(CASE WHEN Distance IS NULL OR Distance <= 0 THEN 1 ELSE 0 END) AS bad_distance,
  SUM(CASE WHEN CRSDepTime IS NULL THEN 1 ELSE 0 END) AS no_sched_dep
FROM airline_analytics.bronze.flights_recent
WHERE Year IN (2020, 2021)

**Zero failures across 10,683,751 rows** on all three testable rules. Combined with the empty
quarantine across the full build, that is ~25M rows examined with no hard failures.

This is the expected result — both sources are curated federal datasets, not scraped data.
The rules are retained as guardrails against future ingestion faults rather than as remediation
for observed problems. The soft-warning tier is where the real defects surfaced.